# PySpark Advanced Level Functions

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/workspace/introduction/raw-data/BigMart Sales.csv")
)

In [0]:
display(df.limit(11))

---

## WHEN-OTHERWISE

In [0]:
"""
when() adalah padanan WHEN ... THEN pada SQL.
otherwise() adalah padanan ELSE.
Gunakan withColumn() untuk membuat atau memperbarui kolom berdasarkan hasil kondisi.
Beberapa kondisi dapat dirangkai dengan .when(...).when(...).otherwise(...).
Spark mengevaluasi kondisi dari atas ke bawah dan berhenti pada kondisi pertama yang bernilai True.
Untuk kondisi gabungan, gunakan operator &, |, dan ~, bukan and, or, atau not
"""

In [0]:
df_when = df.withColumn("flag_veg", when(col("Item_Type") == "Meat", "Non-Vegan").otherwise("Vegan"))
display(df_when.limit(5))

In [0]:
df_when = df.withColumn(
    "flag_resource",
    when( (col("Item_Type") != "Meat") & (col("Item_MRP") < 100) , "Inexpensive-Vegan" )
    .when( (col("Item_Type") != "Meat") & (col("Item_MRP") > 100) , "Expensive-Vegan" ) 
    .otherwise("Non-Vegan")
)

display(df_when.limit(7))

---

## JOIN

In [0]:
"""
df1.join(df2, on, how)

df_customer.join(
    df_order,
    df_customer.customer_id == df_order.customer_id,
    "left_anti"
)

| Jenis Join       | Hasil                                                                   |
| ---------------- | ----------------------------------------------------------------------- |
| `inner`          | Hanya data yang cocok di kedua DataFrame (mirip left_semi)              |
| `left`           | Semua data kiri + data kanan yang cocok                                 |
| `right`          | Semua data kanan + data kiri yang cocok                                 |
| `outer` / `full` | Semua data dari kedua DataFrame                                         |
| `left_semi`      | Baris dari kiri yang memiliki pasangan di kanan; hanya kolom kiri       |
| `left_anti`      | Baris dari kiri yang tidak memiliki pasangan di kanan; hanya kolom kiri |

"""

In [0]:
waifu = [
    (1, "Jinhsi", "e1"),
    (2, "Shorekeeper", "e1"),
    (3, "Carlotta", "e2"),
    (4, "Cartethya", "e3"),
    (5, "Hiyuki", "e2"),
    (6, "Suisui", "e2")
]
schema = "id INT, waifu_name STRING, attr_id STRING"
df_waifu = spark.createDataFrame(waifu, schema)

attribute = [
    ("e1", "Spectro", "Menang 50/50"),
    ("e2", "Glacio", "Menang 50/50"),
    ("e3", "Aero", "Menang 50/50"),
    ("e4", "Havoc", "Menang 50/50"),
    ("e5", "Mixed", "Menang 50/50")
]
schema = "attr_id STRING, attr_name STRING, gacha STRING"
df_attr = spark.createDataFrame(attribute, schema)

In [0]:
display(df_waifu)

In [0]:
display(df_attr)

### Inner Join

In [0]:
df_inner = df_waifu.join(
    df_attr,
    on="attr_id",
    how="inner"
)
display(df_inner)

### Left Join

In [0]:
df_left = df_waifu.join(
    df_attr,
    df_waifu.attr_id == df_attr.attr_id,
    how="left"
)
display(df_left)

### Full Join

In [0]:
df_full = df_waifu.join(
    df_attr,
    on="attr_id",
    how="outer"
).orderBy("attr_id")
display(df_full)

### Left Anti Join

In [0]:
df_left_anti = df_waifu.join(
    df_attr,
    on="attr_id",
    how="left_anti"
)
display(df_left_anti)

---

## WINDOW FUNCTIONS

In [0]:
"""
Perbandingan dengan SQL dan Pyspark

SELECT *,
       ROW_NUMBER() OVER (PARTITION BY Store ORDER BY Sales DESC) AS rn
FROM sales;

-------------------------------------------------------

window = Window.partitionBy("Store").orderBy(col("Sales").desc())

df.withColumn(
    "rn",
    row_number().over(window)
)

-------------------------------------------------------

- Window Function menghitung nilai berdasarkan sekumpulan baris yang berkaitan, tetapi tidak mengurangi jumlah baris.
- partitionBy() membagi data ke dalam kelompok, mirip konsep GROUP BY.
- orderBy() menentukan urutan baris di dalam setiap kelompok.
- .over(window) menerapkan fungsi ke window yang telah didefinisikan.
- Gunakan groupBy() jika ingin menghasilkan ringkasan per grup. Gunakan Window Function jika ingin mempertahankan setiap baris sambil menambahkan hasil perhitungan sebagai kolom baru.

| Fungsi         | Kegunaan                                                                |
| -------------- | ----------------------------------------------------------------------- |
| `row_number()` | Nomor urut unik dalam setiap partition                                  |
| `rank()`       | Ranking dengan nilai yang sama mendapat peringkat sama dan ada lompatan |
| `dense_rank()` | Ranking tanpa lompatan                                                  |
| `lag()`        | Nilai dari baris sebelumnya                                             |
| `lead()`       | Nilai dari baris berikutnya                                             |
| `sum()`        | Total per partition atau running total                                  |
| `avg()`        | Rata-rata per partition                                                 |
| `min()`        | Nilai minimum per partition                                             |
| `max()`        | Nilai maksimum per partition                                            |

"""

In [0]:
display(df.limit(7))

### Row Number

In [0]:
from pyspark.sql.window import Window

In [0]:
# Membuat surrogate_key (identifier unik) pada DataFrame

df_window = df.withColumn(
    "Row_Number",
    row_number().over(Window.orderBy("Item_Identifier"))
)
display(df_window)

### Rank & Dense Rank

In [0]:
window = Window.partitionBy("Item_Type").orderBy("Item_Outlet_Sales")

df_rank = (
    df.withColumn("Rank", rank().over(window))
    .withColumn("Dense_Rank", dense_rank().over(window))
)

display(df_rank)

### Cumulative Sum (Running Total)

In [0]:
window = Window.partitionBy("Item_Type").orderBy("Outlet_Establishment_Year")

df_runtal = df.withColumn(
    "Running_Total",
    sum("Item_Outlet_Sales").over(window)
)

display(df_runtal)

In [0]:
df_runtal_fix = df.withColumn(
    "Running_Total",
    sum("Item_Outlet_Sales").over(
        Window.partitionBy("Item_Type")
        .orderBy("Outlet_Establishment_Year")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
)

display(df_runtal_fix)

---

## USER DEFINED FUNCTION (UDF)

In [0]:
"""
Gunakan UDF jika:
- Logika bisnis tidak bisa dibuat dengan function bawaan Spark.
- Perlu menggunakan algoritma Python yang sederhana dan spesifik.

Hindari UDF jika logikanya bisa dibuat menggunakan:
- when
- otherwise
- regexp_replace
- concat
- upper
- lower
- substring
- split
- array
- transform
- aggregate
- function bawaan Spark lainnya
"""

In [0]:
# Step 1: Define the function (python)
def my_function(x):
    return x*2

In [0]:
# Step 2: Convert the python function into Pyspark UDF
my_udf = udf(my_function)

# Step 3: Apply the UDF to the DataFrame
df_my_udf = df.withColumn("User_Function", my_udf("Item_Type"))

display(df_my_udf.limit(7))
        